# `03b_extra_baselines_opt_67b.ipynb` — MIND + Perplexity only (OPT-6.7B)

Runs **only the 2 remaining baselines** (MIND, Perplexity) on the **same 10 datasets at `QUICK_EVAL_N = 350` first-N** as `03_baselines_sota_opt_67b.ipynb`, so the numbers merge directly with the 4-baseline run.

**Inputs (Kaggle):** add `kaggle_opt_67b_dataset_full.json` and `eval_*.parquet` as Datasets. **Accelerator: GPU T4 × 2** (OPT-6.7B is 7B-class). Output: `kaggle_opt_67b_extra_baselines_results.json`.

- **MIND** (Su et al. 2024): MLP probe on the last-layer last-token hidden state (the canonical embedding already in `dataset_full`). Supervised.
- **Perplexity**: mean token negative-log-likelihood of the text; unsupervised, threshold-free (rank-normalised at metric time).

In [ ]:
# ---- BLOCK 0: Kaggle setup ----
import os, glob, shutil
WORK='/kaggle/working'; os.chdir(WORK)
def bring_in(pat):
    out=[]
    for src in glob.glob('/kaggle/input/**/'+pat, recursive=True):
        dst=os.path.join(WORK, os.path.basename(src))
        if os.path.abspath(src)!=os.path.abspath(dst):
            if not os.path.exists(dst): shutil.copy(src,dst)
            out.append(os.path.basename(src))
    return sorted(out)
print('dataset_full:', bring_in('kaggle_opt_67b_dataset_full.json'))
print('eval parquets:', bring_in('eval_*.parquet'))

In [ ]:
# ---- BLOCK 1: setup + load dataset_full + lazy model loader ----
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','sentence-transformers'],check=False)
import json, time, gc, random, numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME='facebook/opt-6.7b'; MODEL_TAG='kaggle_opt_67b'
DTYPE=torch.bfloat16 if torch.cuda.is_available() else torch.float32
SEED=42; TEST_FRAC=0.2; QUICK_EVAL_N=350; MAX_GEN_NEW=48; FEAT_MAX_LEN=512
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATH_FULL=f'{MODEL_TAG}_dataset_full.json'
records=json.load(open(PATH_FULL))
print(f'{len(records)} records, hidden_dim {len(records[0]["embedding"])}')
_LLM={'tok':None,'model':None}
def need_llm():
    if _LLM['model'] is not None: return _LLM
    print(f'Loading {MODEL_NAME} ...'); t0=time.perf_counter()
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE,
        device_map='auto' if torch.cuda.is_available() else None)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad=False
    _LLM['tok'],_LLM['model']=tok,mdl
    print(f'  loaded in {time.perf_counter()-t0:.1f}s')
    return _LLM

In [ ]:
# ---- BLOCK 2: train MIND (MLP on canonical last-layer embedding) ----
class MINDProbe(nn.Module):
    def __init__(self,d):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,x): return self.net(x)
X=np.array([r['embedding'] for r in records],dtype=np.float32)
y=np.array([r['label'] for r in records],dtype=np.float32)
idx=list(range(len(records))); random.Random(SEED).shuffle(idx)
sp=int((1-TEST_FRAC)*len(idx)); tr,te=np.array(idx[:sp]),np.array(idx[sp:])
Xtr=torch.from_numpy(X[tr]).to(device); ytr=torch.from_numpy(y[tr]).to(device)
Xte=torch.from_numpy(X[te]).to(device); yte=torch.from_numpy(y[te]).to(device)
mind_eval=MINDProbe(X.shape[1]).to(device)
opt=torch.optim.Adam(mind_eval.parameters(),lr=5e-4); lf=nn.BCEWithLogitsLoss()
for ep in range(10):
    mind_eval.train(); perm=torch.randperm(Xtr.shape[0],device=device)
    for i in range(0,Xtr.shape[0],32):
        b=perm[i:i+32]; opt.zero_grad(); l=lf(mind_eval(Xtr[b]).squeeze(-1),ytr[b]); l.backward(); opt.step()
mind_eval.eval()
with torch.no_grad():
    acc=float(((torch.sigmoid(mind_eval(Xte).squeeze(-1))>0.5).float()==yte).float().mean().item())
torch.save({'model_state':mind_eval.state_dict(),'input_dim':int(X.shape[1])}, f'{MODEL_TAG}_mind_best.pth')
print(f'MIND trained on dim {X.shape[1]}, held-out acc {acc:.4f}')

In [ ]:
# ---- BLOCK 3: scorers (MIND+Perplexity in one forward) + metrics + MiniLM ----
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, brier_score_loss, confusion_matrix
from sentence_transformers import SentenceTransformer
LL=need_llm(); tok,model=LL['tok'],LL['model']
scorer=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=str(device))
def score_match(gen, golds, threshold=0.5):
    if not golds: return 0
    if isinstance(golds,str): golds=[golds]
    golds=[g for g in golds if g]
    if not golds: return 1
    e=scorer.encode([gen]+list(golds), convert_to_numpy=True, normalize_embeddings=True)
    return 0 if float((e[0]@e[1:].T).max())>=threshold else 1
@torch.no_grad()
def generate_short_answer(prompt, max_new=MAX_GEN_NEW):
    mp=getattr(model.config,'max_position_embeddings',2048) or 2048
    enc=tok(prompt,return_tensors='pt',truncation=True,max_length=max(mp-max_new,256)).to(model.device)
    out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc.input_ids.shape[1]:],skip_special_tokens=True).strip()
@torch.no_grad()
def mind_and_ppl(text):
    cap=min(getattr(model.config,'max_position_embeddings',2048) or 2048, FEAT_MAX_LEN)
    enc=tok(text.strip(),return_tensors='pt',truncation=True,max_length=cap).to(model.device)
    ids=enc.input_ids
    out=model(**enc, output_hidden_states=True, use_cache=False)
    h=out.hidden_states[-1][0,-1,:].float()
    prob=float(torch.sigmoid(mind_eval(h.unsqueeze(0).to(device)).squeeze(-1))[0].item())
    ppl=float('nan')
    if ids.shape[1]>=2:
        lg=out.logits[0,:-1,:].float(); ppl=float(F.cross_entropy(lg, ids[0,1:], reduction='mean').item())
    return prob, ppl
def _metrics(y,p,pr):
    y=np.asarray(y); p=np.asarray(p); pr=np.nan_to_num(np.asarray(pr,dtype=float),nan=0.5,posinf=1.0,neginf=0.0)
    if len(y)==0: return {'n':0}
    try: auc=float(roc_auc_score(y,pr)) if len(set(y.tolist()))>1 else float('nan')
    except ValueError: auc=float('nan')
    pr_,rc_,f1_,_=precision_recall_fscore_support(y,p,average='binary',zero_division=0)
    cm=confusion_matrix(y,p,labels=[0,1])
    return {'n':int(len(y)),'accuracy':float(accuracy_score(y,p)),'precision':float(pr_),'recall':float(rc_),
            'f1':float(f1_),'auc_roc':auc,'brier':float(brier_score_loss(y,pr)),
            'cm':{'tn':int(cm[0,0]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),'tp':int(cm[1,1])}}
def _score_to_metrics(scores,labels):
    scores=np.asarray(scores,dtype=float); labels=np.asarray(labels)
    finite=~np.isnan(scores)
    if finite.sum()<2: return {'n':int(len(labels))}
    ranks=np.argsort(np.argsort(scores[finite])).astype(float)
    ps=np.full_like(scores,0.5); ps[finite]=ranks/max(len(ranks)-1,1)
    return _metrics(labels,(ps>0.5).astype(int),ps)
print('scorers ready')

In [ ]:
# ---- BLOCK 4: load 10 eval datasets (350 first-N, local parquet) ----
from datasets import load_dataset, Dataset as HFDataset
import pyarrow.parquet as pq
def load_eval(label, *tries):
    for tpl in ['eval_{}.parquet','/kaggle/input/dissertation-eval-datasets/eval_{}.parquet','/kaggle/working/eval_{}.parquet']:
        p=tpl.format(label)
        if os.path.exists(p):
            ds=HFDataset.from_pandas(pq.read_table(p).to_pandas())
            ds=ds.select(range(min(QUICK_EVAL_N,len(ds)))); print(f'  {label}: {len(ds)} (LOCAL)'); return ds
    for fn in tries:
        try:
            ds=fn(); ds=ds.select(range(min(QUICK_EVAL_N,len(ds)))); print(f'  {label}: {len(ds)} (HF)'); return ds
        except Exception as e: print(f'  [warn] {label}: {e}')
    return None
D={}
D['truthfulqa']=load_eval('truthfulqa', lambda: load_dataset('truthfulqa/truthful_qa','generation',split='validation'))
D['triviaqa']=load_eval('triviaqa', lambda: load_dataset('mandarjoshi/trivia_qa','rc.nocontext',split='validation'))
D['coqa']=load_eval('coqa', lambda: load_dataset('stanfordnlp/coqa',split='validation'))
D['tydiqa']=load_eval('tydiqa', lambda: load_dataset('google-research-datasets/tydiqa','secondary_task',split='validation'))
for cfg,lab in [('qa','halueval_qa'),('summarization','halueval_summ'),('dialogue','halueval_dialog')]:
    D[lab]=load_eval(lab, lambda c=cfg: load_dataset('pminervini/HaluEval',c,split='data'))
D['nq_open']=load_eval('nq_open', lambda: load_dataset('google-research-datasets/nq_open',split='validation'))
D['hotpotqa']=load_eval('hotpotqa', lambda: load_dataset('hotpotqa/hotpot_qa','distractor',split='validation',trust_remote_code=True))
D['popqa']=load_eval('popqa', lambda: load_dataset('akariasai/PopQA',split='test'))
print('loaded:', {k:(len(v) if v is not None else 0) for k,v in D.items()})

In [ ]:
# ---- BLOCK 5: multi-task eval (MIND + Perplexity) + dump ----
from tqdm.auto import tqdm
import json as _json
def _hotpot_pr(s):
    try: ctx=' '.join([' '.join(p) for p in s['context']['sentences'][:3]])[:600]
    except Exception: ctx=''
    return f'Context: {ctx} Q: {s["question"]} A:'
def _popqa_gold(s):
    pa=s.get('possible_answers','[]')
    if isinstance(pa,str):
        try: pa=_json.loads(pa)
        except Exception: pa=[pa]
    return pa if isinstance(pa,list) else [pa]
TASKS=[
 ('truthfulqa','open',lambda s:f'Answer the question concisely. Q: {s["question"]} A:',lambda s:s.get('correct_answers',[]),None,None),
 ('triviaqa','open',lambda s:f'Answer the question concisely. Q: {s["question"]} A:',lambda s:list(s['answer'].get('aliases',[]))+[s['answer'].get('value','')],None,None),
 ('coqa','open',lambda s:f'Context: {s["story"][:600]} Q: {(s["questions"][0] if s["questions"] else "")} A:',lambda s:(s['answers']['input_text'][0] if s['answers']['input_text'] else ''),None,None),
 ('tydiqa','open',lambda s:f'Context: {s["context"][:600]} Q: {s["question"]} A:',lambda s:s.get('answers',{}).get('text',[]),None,None),
 ('halueval_qa','he',lambda s:f'Context: {s["knowledge"][:400]} Q: {s["question"]} A:',None,'right_answer','hallucinated_answer'),
 ('halueval_summ','he',lambda s:f'{s["document"][:500]} Summary:',None,'right_summary','hallucinated_summary'),
 ('halueval_dialog','he',lambda s:f'Knowledge: {s["knowledge"][:300]}\nDialogue: {s["dialogue_history"][:300]}\n[Assistant]:',None,'right_response','hallucinated_response'),
 ('nq_open','open',lambda s:f'Answer the question concisely. Q: {s["question"]} A:',lambda s:s.get('answer',[]) if isinstance(s.get('answer'),list) else [s.get('answer','')],None,None),
 ('hotpotqa','open',_hotpot_pr,lambda s:[s.get('answer','')],None,None),
 ('popqa','open',lambda s:f'Answer the question concisely. Q: {s["question"]} A:',_popqa_gold,None,None),
]
multitask={'MIND':{},'Perplexity':{}}; fails={}
_t0=time.perf_counter()
for name,kind,pf,gf,rk,wk in TASKS:
    ds=D.get(name)
    if ds is None: continue
    yy=[]; mind_pr=[]; ppl=[]; f=0
    for s in tqdm(ds, desc=name):
        try:
            if kind=='open':
                pr=pf(s); gen=generate_short_answer(pr); lab=score_match(gen, gf(s))
                mp_,pp_=mind_and_ppl((pr+' '+gen).strip()); yy.append(lab); mind_pr.append(mp_); ppl.append(pp_)
            else:
                pr=pf(s)
                for ak,gl in [(rk,0),(wk,1)]:
                    mp_,pp_=mind_and_ppl((pr+' '+s[ak]).strip()); yy.append(gl); mind_pr.append(mp_); ppl.append(pp_)
        except Exception: f+=1; continue
    multitask['MIND'][name]=_metrics(yy,[int(p>0.5) for p in mind_pr],mind_pr)
    multitask['Perplexity'][name]=_score_to_metrics(np.array(ppl),np.array(yy))
    fails[name]=f
    a1=multitask['MIND'][name].get('auc_roc',float('nan')); a2=multitask['Perplexity'][name].get('auc_roc',float('nan'))
    print(f'  {name:14s} MIND AUROC={a1:.3f}  Perplexity AUROC={a2:.3f}  (n={multitask["MIND"][name].get("n",0)}, fails={f})')
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
hid=len(records[0]['embedding'])
spec={'MIND':{'paradigm':'supervised probe','feature':'last-layer last-token hidden','dim':hid,'justification':'Su et al. 2024 - the method this work extends'},
      'Perplexity':{'paradigm':'unsupervised (score)','feature':'mean token NLL','dim':1,'justification':'classic uncertainty baseline'}}
DIAG={'model_tag':MODEL_TAG,'model_name':MODEL_NAME,'quick_eval_n':QUICK_EVAL_N,
      'baselines':{b:{'multitask':multitask[b]} for b in ['MIND','Perplexity']},
      'baseline_feature_spec':spec,'n_eval_failures_per_dataset':fails,
      'eval_seconds':round(time.perf_counter()-_t0,1)}
with open(f'{MODEL_TAG}_extra_baselines_results.json','w') as fobj: json.dump(DIAG,fobj,indent=2)
print('\nwrote', f'{MODEL_TAG}_extra_baselines_results.json', '  (merge MIND+Perplexity into your 4-baseline table)')
try:
    from IPython.display import FileLink; display(FileLink(f'{MODEL_TAG}_extra_baselines_results.json'))
except Exception: pass